In [1]:
import sys, os
import xarray as xr
import pandas as pd
import numpy as np
import itertools ## need this for the cbarticks

import cartopy.crs as ccrs
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import cartopy.feature as cfeature
from matplotlib.colorbar import Colorbar # different way to handle colorbar

sys.path.append('modules')
from plotter import draw_basemap
import cw3ecmaps as ccmaps

In [2]:
def map_PFDF_with_AR(lat, lon, event_date, event_name, ds):
    # Set up projection
    datacrs = ccrs.PlateCarree()  ## the projection the data is in
    mapcrs = ccrs.PlateCarree() ## the projection you want your map displayed in
    ext = [-140., -110., 20, 50]

    # Set tick/grid locations
    tx = 10
    ty = 5
    dx = np.arange(ext[0],ext[1]+tx,tx)
    dy = np.arange(ext[2],ext[3]+ty,ty)

    nrows = 3
    ncols = 2
    
    ## Use gridspec to set up a plot with a series of subplots that is
    ## n-rows by n-columns
    gs = GridSpec(nrows, ncols, height_ratios=[1, 1, 0.05], width_ratios = [1, 1], wspace=0.01, hspace=0.2)
    ## use gs[rows index, columns index] to access grids
    
    fig = plt.figure(figsize=(8, 9.))
    fig.dpi = 300
    fname = f'figs/{event_date}-{event_name}'
    fmt = 'png'

    ## loop through time step
    row_idx = [0, 0, 1, 1]
    col_idx = [0, 1, 0, 1]
    llats = [True, False]*2
    blons = [False, False, True, True]
    date_lst = pd.date_range(event_date, periods=4, freq="6h")
    tmp = ds.sel(time=date_lst)
    
    for i, date, in enumerate(tmp.time.values):
        ax = fig.add_subplot(gs[row_idx[i], col_idx[i]], projection=mapcrs)
    
        ax = draw_basemap(ax, extent=ext, xticks=dx, yticks=dy,
                          left_lats=llats[i], right_lats=False, bottom_lons=blons[i])
        ax.set_extent(ext, datacrs)
        ax.add_feature(cfeature.STATES, edgecolor='0.4', linewidth=0.8)

        # add titles
        title = pd.to_datetime(date).strftime('%H UTC %d %b %Y')
        ax.set_title(title, loc='left')
    
        # add point location of PFDF
        ax.plot(lon, lat, 'ko', markersize=3, transform=datacrs, zorder=201)
    
        ## add tARget contours
        cmap, norm, bnds, cbarticks, cbarlbl = ccmaps.cmap('arscale') # get cmap from our custom function
        plot_da = tmp.sel(time=date)
        cf = ax.contourf(plot_da.longitude.values, plot_da.latitude.values, plot_da.values, transform=datacrs,
                         levels=bnds, cmap=cmap, norm=norm, alpha=0.9)

    # Add color bar
    cbax = plt.subplot(gs[-1, :]) # colorbar axis (last row, all columns)
    cbarticks = list(itertools.compress(bnds, cbarticks)) ## this labels the cbarticks based on the cmap dictionary
    cb = Colorbar(ax = cbax, mappable = cf, orientation = 'horizontal', ticklocation = 'bottom', ticks=cbarticks)
    cb.set_label(cbarlbl, fontsize=11)
    cb.ax.tick_params(labelsize=12)

    fig.savefig('%s.%s' %(fname, fmt), bbox_inches='tight', dpi=fig.dpi, transparent=True)
    # plt.show()
    fig.clf()

In [3]:
# --- Create dataframe with information about 5 events ---
events = [
    {"lat": 34.06, "lon": -118.64, "event_date": "2025-02-13 12", "event_name": "Palisades"},
    {"lat": 34.16, "lon": -118.09, "event_date": "2025-02-13 12", "event_name": "Eaton"},
    {"lat": 33.87, "lon": -116.95, "event_date": "2025-03-13 00", "event_name": "Record"},
    {"lat": 34.16, "lon": -118.05, "event_date": "2021-12-14 00", "event_name": "Bobcat"},
    {"lat": 34.09, "lon": -118.95, "event_date": "2021-12-30 00", "event_name": "Woolsey"},
]

df = pd.DataFrame(events)

# --- read ARscale / ARDT data --- 
fname = "data/CDR_ERA5_ARScale_v2.0_*.nc"
ds = xr.open_mfdataset(fname)
ds

<xarray.Dataset> Size: 10GB
Dimensions:               (time: 360, latitude: 721, longitude: 1440)
Coordinates:
  * time                  (time) datetime64[ns] 3kB 2021-12-01 ... 2025-03-31...
  * latitude              (latitude) float32 3kB 90.0 89.75 ... -89.75 -90.0
  * longitude             (longitude) float32 6kB 0.0 0.25 0.5 ... 359.5 359.8
Data variables:
    ivt                   (time, latitude, longitude) float32 1GB dask.array<chunksize=(124, 721, 1440), meta=np.ndarray>
    ARs                   (time, latitude, longitude) float32 1GB dask.array<chunksize=(124, 721, 1440), meta=np.ndarray>
    ARScale               (time, latitude, longitude) int32 1GB dask.array<chunksize=(124, 721, 1440), meta=np.ndarray>
    ARScale_ARDT          (time, latitude, longitude) int32 1GB dask.array<chunksize=(124, 721, 1440), meta=np.ndarray>
    EnhancedARScale       (time, latitude, longitude) int32 1GB dask.array<chunksize=(124, 721, 1440), meta=np.ndarray>
    EnhancedARScale_ARDT  (time, latitude, longitude) int32 1GB dask.array<chunksize=(124, 721, 1440), meta=np.ndarray>
    TP                    (time, latitude, longitude) float32 1GB dask.array<chunksize=(124, 721, 1440), meta=np.ndarray>
Attributes:
    title:        IVT and AR scale of detected ARs from ERA5
    source:       ECMWF ERA5
    description:  IVT, AR Scale, and AR Duration

In [4]:

for index, row in df.iterrows():
    event_name = row['event_name']
    event_date = row['event_date']
    lat = round(row['lat'])
    lon = round(row['lon'])
    print(event_date, lat, lon)
    
    map_PFDF_with_AR(lat, lon, event_date, event_name, ds.ARScale_ARDT)

2025-02-13 12 34 -119
2025-02-13 12 34 -118
2025-03-13 00 34 -117
2021-12-14 00 34 -118
2021-12-30 00 34 -119


<Figure size 2400x2700 with 0 Axes>

<Figure size 2400x2700 with 0 Axes>

<Figure size 2400x2700 with 0 Axes>

<Figure size 2400x2700 with 0 Axes>

<Figure size 2400x2700 with 0 Axes>